# Limpieza y Normalización de Datos

Este notebook procesa los datos combinados obtenidos en la etapa de extracción para asegurar consistencia, corrigiendo nombres, tratando valores nulos y agrupando los registros por jugador.

## 1. Importación y Configuración Inicial
Importamos `pandas` y `numpy`, y definimos las constantes como las rutas de directorios y la lista de columnas numéricas (minutos, goles, asistencias, tarjetas).


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "web_scraping" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "web_scraping" / "processed"

NUMERIC_COLUMNS = [
    "minutes_played",
    "goals",
    "assists",
    "yellow_cards",
    "red_cards",
]


## 2. Mapeo de Equipos y Filtrado
Se establecen diccionarios para unificar las diferentes formas en las que están escritos los nombres de los equipos de LaLiga (`_TEAM_MAP`) y un conjunto de equipos extranjeros o descartados (`_NON_LALIGA_TEAMS`). Las funciones `normalize_team_names` y `drop_non_laliga_rows` se encargan de aplicar estas reglas.


In [2]:
_TEAM_MAP = {
    "FC Barcelona": "Barcelona",
    "RCD Espanyol de Barcelona": "Espanyol",
    "Espanyol": "Espanyol",
    "Atlético de Madrid": "Atlético Madrid",
    "Atletico": "Atlético Madrid",
    "Athletic Bilbao": "Athletic Club",
    "CA Osasuna": "Osasuna",
    "Sevilla FC": "Sevilla",
    "Cádiz CF": "Cádiz",
    "Cadiz": "Cádiz",
    "Elche CF": "Elche",
    "Getafe CF": "Getafe",
    "Girona FC": "Girona",
    "Granada CF": "Granada",
    "Levante UD": "Levante",
    "RCD Mallorca": "Mallorca",
    "RC Celta": "Celta de Vigo",
    "Celta Vigo": "Celta de Vigo",
    "UD Almería": "Almería",
    "Almeria": "Almería",
    "UD Las Palmas": "Las Palmas",
    "Deportivo Alavés": "Alavés",
    "Real Valladolid CF": "Valladolid",
    "Real Valladolid": "Valladolid",
    "Valencia CF": "Valencia",
    "Villarreal CF": "Villarreal",
    "CD Leganés": "Leganés",
    "AD Alcorcón": "Alcorcón",
    "CD Mirandés": "Mirandés",
    "CD Tenerife": "Tenerife",
    "CF Fuenlabrada": "Fuenlabrada",
    "Málaga CF": "Málaga",
    "SD Amorebieta": "Amorebieta",
    "SD Eibar": "Eibar",
    "SD Huesca": "Huesca",
    "SD Ponferradina": "Ponferradina",
    "UD Ibiza": "Ibiza",
    "Real Zaragoza": "Zaragoza",
    "R. Sociedad B": "Real Sociedad B",
}

_NON_LALIGA_TEAMS = frozenset({
    "A Villa",
    "Brighton",
    "Chelsea",
    "Fiorentina",
    "LOSC",
    "Newcastle",
    "Spezia",
    "Spurs",
    "Udinese",
})


def normalize_team_names(series: pd.Series) -> pd.Series:
    result = series.astype("string").str.strip()
    return result.replace(_TEAM_MAP)

def drop_non_laliga_rows(data: pd.DataFrame, team_column: str = "team") -> pd.DataFrame:
    normalized_teams = normalize_team_names(data[team_column])
    mask = ~normalized_teams.isin(_NON_LALIGA_TEAMS)
    return data.loc[mask].copy()


## 3. Limpieza del Conjunto de Datos
Cargamos los datos combinados, eliminamos espacios en blanco y valores nulos en los nombres de los jugadores. Además, convertimos las estadísticas a tipos numéricos, rellenamos valores faltantes, y aplicamos la normalización y filtrado de los equipos definidos anteriormente.


In [3]:
data = pd.read_csv(PROCESSED_DIR / "datos_combinados.csv")
data = data.replace(r"^\s*$", np.nan, regex=True)
data["player_name"] = data["player_name"].astype("string").str.strip()
data = data.dropna(subset=["player_name"])

for column in NUMERIC_COLUMNS:
    data[column] = pd.to_numeric(data[column], errors="coerce").fillna(0.0)

for column in ("team", "position", "season"):
    data[column] = data[column].astype("string").fillna("Unknown").replace("", "Unknown")

data["team"] = normalize_team_names(data["team"])
data = drop_non_laliga_rows(data)


## 4. Agrupación y Guardado
Agrupamos los registros para unificar a los jugadores que puedan aparecer varias veces. Para la posición, calculamos la moda estadística, y para las métricas numéricas, sumamos los totales. Finalmente, exportamos el dataset limpio a un archivo CSV.


In [4]:
def mode_or_unknown(values: pd.Series) -> str:
    modes = values.dropna().mode()
    return str(modes.iloc[0] if not modes.empty else "Unknown")

cleaned_data = (
    data.groupby(["player_name", "team", "season"], as_index=False)
    .agg(
        position=("position", mode_or_unknown),
        **{column: (column, "sum") for column in NUMERIC_COLUMNS},
    )
)
cleaned_data.to_csv(PROCESSED_DIR / "datos_limpios.csv", index=False)
